In [11]:
# Ordner labs/ finden und zum Arbeitsordner machen: Das Paket common und die Daten liegen dort.
# Nötig, wenn der Kernel woanders startet (PyCharm, VS Code mit Arbeitsordner im Pack-Root, Quelle in labs/_src/).
import os, sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "labs", *Path.cwd().parents]:
    if (_p / "common" / "__init__.py").exists():
        os.chdir(_p)
        sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError("Ordner labs/ nicht gefunden. Öffnen Sie das Notebook aus dem Ordner labs/ des Kursmaterials.")

import json
import logging
import time
import warnings

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import mlflow
import numpy as np
import pandas as pd

from common.paths import ARTIFACTS_DIR, offline

offline()  # vor dem ersten Laden eines Modells: die Labs laufen ohne Netzzugriff

from common.contract import GERMAN_QUERIES, NEW_DELIVERY, check_contract, known_word_share
from common.data import dataset_hash, load_banking77
from common.encoder_eval import (corpus_embeddings, hits_table, ndcg_per_query, package_versions,
                                 paired_bootstrap, query_embeddings)
from common.evalset import build_evalset
from common.interactive import explore
from common.metrics import retrieval_metrics, top_k
from common.models import EMBEDDING_MODELS, encode, load_encoder, size_mb, timed_encode
from common.plotstyle import use_course_style
from common.tracking import setup_mlflow

use_course_style()
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)  # Hinweis beim Laden von rohem DistilBERT
SEED, DEVICE, BATCH_SIZE = 1, "cpu", 128
OUT_DIR = ARTIFACTS_DIR / "lab1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

logging.disable(logging.INFO)  # das erste Anlegen der Datenbank schreibt sonst viele Migrationszeilen
tracking_uri = setup_mlflow("lab1")
logging.disable(logging.NOTSET)
logging.getLogger("mlflow").setLevel(logging.ERROR)

train, test, intents = load_banking77()
print(f"{len(train)} Trainings- und {len(test)} Testanfragen, {len(intents)} Anliegen")

9993 Trainings- und 3076 Testanfragen, 77 Anliegen


In [3]:
minilm = load_encoder("minilm-l6", device=DEVICE)
distilbert = load_encoder("distilbert-raw", device=DEVICE)

a = "I am still waiting on my card?"   # card_arrival
b = "My card isn't here yet."          # card_arrival
c = "What exchange rate is it?"        # exchange_rate
ANFRAGE_AUS_DEM_RAUM = "My new card has still not arrived"  # hier eine Anfrage aus der Gruppe eintragen

table = {}
for name, model in [("minilm-l6", minilm), ("distilbert-raw", distilbert)]:
    emb = encode(model, [a, b, c, ANFRAGE_AUS_DEM_RAUM])
    sim = emb @ emb.T  # normierte Vektoren: Skalarprodukt = Cosine
    table[name] = {
        "a, b (gleiches Anliegen)": sim[0, 1],
        "a, c (anderes Anliegen)": sim[0, 2],
        "Abstand": sim[0, 1] - sim[0, 2],
        "Raum, a": sim[3, 0],
        "Raum, c": sim[3, 2],
    }
cosines = pd.DataFrame(table)
print(f"Größe: minilm-l6 {size_mb(minilm):.0f} MB, distilbert-raw {size_mb(distilbert):.0f} MB")
cosines.round(3)

Größe: minilm-l6 91 MB, distilbert-raw 265 MB


,minilm-l6,distilbert-raw
"a, b (gleiches Anliegen)",0.702,0.872
"a, c (anderes Anliegen)",0.065,0.733
Abstand,0.637,0.139
"Raum, a",0.730,0.879
"Raum, c",0.054,0.657


In [7]:
# Eine Kollegin hat MiniLM letzte Woche kurz getestet. Metrik war Hit@1. 

import pandas as pd
from sentence_transformers import SentenceTransformer

train_df = pd.read_parquet("data/banking77/train.parquet")
test_df = pd.read_parquet("data/banking77/test.parquet")
model = SentenceTransformer("models/minilm-l6", device="cpu")

corpus = train_df.iloc[:3000]  # Teilmenge, damit es schnell geht
queries = test_df.sample(200)
corpus_emb = model.encode(corpus["text"].tolist(), normalize_embeddings=True)
query_emb = model.encode(queries["text"].tolist(), normalize_embeddings=True)

best = (query_emb @ corpus_emb.T).argmax(axis=1)
hit_rate = (corpus["label"].to_numpy()[best] == queries["label"].to_numpy()).mean()
print(f"Trefferquote Top-1: {hit_rate:.2f}")

Trefferquote Top-1: 0.26


In [ ]:
# Im Mittel 0.3, zwischen 0.24 und 0.37 
# Ihr Vorschlag ist: models/minilm-l6 aussortieren 

# Batch-Pfad und Echtzeit-Pfad

In [12]:
index_emb, seconds = timed_encode(minilm, train["text"].tolist(), BATCH_SIZE)
rate = len(index_emb)/seconds
hours_10m = 10_000_000 / rate / 3600 
print(f"Durchsatz: {rate}, 10 Mio. docs: {hours_10m}")


Durchsatz: 2976.0194869220063, 10 Mio. docs: 0.9333869586488283


In [15]:
samples = test["text"].sample(50, random_state=SEED).tolist()
encode_ms, search_ms = [], []
for text in samples:
    t0 = time.perf_counter()
    vector = encode(minilm, [text], batch_size=1)
    t1 = time.perf_counter()
    hits = top_k(vector, index_emb, k=10)
    search_ms.append((time.perf_counter() - t1) * 1e3)
    encode_ms.append((t1 - t0) * 1e3)

print(f"Encoder je Anfrage: Median {np.median(encode_ms):.1f} ms")
print(f"Bei 1000 Anfragen aus dem Durchsatz wären es {1e3 /rate:.2f} ms je Text")
print(f"Suche je Anfrage: Median {np.median(search_ms):.1f} ms")

Encoder je Anfrage: Median 3.9 ms
Bei 1000 Anfragen aus dem Durchsatz wären es 0.34 ms je Text
Suche je Anfrage: Median 0.2 ms


# Sauber bewerten und festhalten

In [25]:
DATA_HASH = dataset_hash(train, test)[:16]


evalset = build_evalset(train,test, n_queries=1000, seed=SEED)
evalset.fingerprint

assert DATA_HASH == "3176e3ac9aef7185" and evalset.fingerprint == "e21d87d6477c7f22"

print(f"Daten-Hash {DATA_HASH}, Eval-Set {evalset.fingerprint} (Seed {evalset.seed}): "
      f"{len(evalset.queries)} Queries gegen {len(evalset.corpus)} Dokumente")
query_emb, query_seconds = timed_encode(minilm, evalset.queries, BATCH_SIZE)
metrics = retrieval_metrics(query_emb, evalset.query_labels, index_emb, evalset.corpus_labels, k=10)
metrics["texts_per_second"] = len(evalset.queries) / query_seconds
metrics["size_mb"] = size_mb(minilm)
pd.Series(metrics).round(3)

Daten-Hash 3176e3ac9aef7185, Eval-Set e21d87d6477c7f22 (Seed 1): 1000 Queries gegen 9993 Dokumente


ndcg_at_10             0.885
mrr_at_10              0.945
precision_at_10        0.874
hit_at_1               0.927
texts_per_second    2551.620
size_mb               90.853
dtype: float64

In [27]:
metrics

{'ndcg_at_10': 0.8854305867034253,
 'mrr_at_10': 0.9451436507936507,
 'precision_at_10': 0.8739,
 'hit_at_1': 0.927,
 'texts_per_second': 2551.6203383883803,
 'size_mb': 90.852864}

In [26]:
EMBEDDING_MODELS["minilm-l6"]

'sentence-transformers/all-MiniLM-L6-v2'

In [28]:
with mlflow.start_run(run_name="minilm-l6-baseline") as run:
    mlflow.log_params({
        "model": "minilm-l6", "hub_id": EMBEDDING_MODELS["minilm-l6"],
        "n_queries": len(evalset.queries), "n_corpus": len(evalset.corpus),
        "batch_size": BATCH_SIZE, "device": DEVICE, 
        "max_seq_length": minilm.max_seq_length, "pooling": minilm[1].pooling_mode, "normalize": True,
    })
    mlflow.log_metrics(metrics)
    mlflow.set_tags({
        "dataset_hash": DATA_HASH, "evalset_fingerprint": evalset.fingerprint, "evalset_seed": SEED,
        **{f"version.{package}": version for package, version in package_versions().items()},
    })
    metrics_path = OUT_DIR / "metrics.csv"
    pd.Series(metrics).to_csv(metrics_path, index=False)
    mlflow.log_artifact(str(metrics_path), artifact_path="analysis")


In [32]:
stored = mlflow.get_run(run.info.run_id)  # aus der Datenbank zurückgelesen 
print(f"Run {stored.info.run_id}:")

print("Tags:", {key: value for key, value in stored.data.tags.items() if not key.startswith("mlflow.")})

Run f5d83bac92cc484fb543f47926081924:
Tags: {'dataset_hash': '3176e3ac9aef7185', 'evalset_fingerprint': 'e21d87d6477c7f22', 'evalset_seed': '1', 'version.sentence-transformers': '5.7.0', 'version.transformers': '4.57.6', 'version.torch': '2.14.1', 'version.mlflow': '2.22.5', 'version.scikit-learn': '1.9.1', 'version.numpy': '2.0.2', 'version.pandas': '2.3.3'}


In [ ]:
per_query = {name: ndcg_per_query(query_embeddings(name, evalset), evalset.query_labels,
                                  corpus_embeddings(name, evalset), evalset.corpus_labels)
             for name in ["minilm-l6", "bge-small"]}

per_query

In [ ]:
per_query["minilm-l6"].mean()

(np.float64(0.8854305867034253), np.float64(1.0))

In [46]:
per_query["bge-small"].mean()


np.float64(0.902646757060652)

In [50]:
diff = per_query["minilm-l6"] - per_query["bge-small"]
np.abs(diff.mean()),diff.std()

(np.float64(0.017216170357226713), np.float64(0.10467081344311306))

In [ ]:
# Champion auswählen 

min_ndcg, champion = 0.85, "minilm-l6"

# def champion_gate(min_lead: float = 0.02) -> str:
    # per_query = {name: ndcg_per_query(query_embeddings(name, evalset), evalset.query_labels,
    #                               corpus_embeddings(name, evalset), evalset.corpus_labels)
    #          for name in ["minilm-l6", "bge-small"]}
    # best = max(per_query, key=per_query.get)
    # diff = per_query["minilm-l6"] - per_query["bge-small"]
    # if min_lead < np.abs(diff.mean()): 



